In [ ]:
import pandas as pd
df = pd.read_csv('./data/건봉국밥_리뷰_250527.csv')
df

In [ ]:
df['세부옵션'].value_counts()

In [ ]:
df = df[df['세부옵션'].str.startswith("100%로 국내산 재료로 만든 전국 3대 국밥")]
df

In [ ]:
import re

def clean_text(text: str) -> str:
    # 특수문자 제거, 영어/숫자 선택적 포함 가능
    text = re.sub(r'[^가-힣\s]', ' ', text)  # 한글과 공백만 남기기
    text = re.sub(r'\s+', ' ', text).strip()  # 공백 정리
    return text

In [ ]:
from konlpy.tag import Okt

okt = Okt()

def tokenize_and_lemmatize(text):
    # 명사, 형용사, 동사 중심으로 추출하고 원형 복원
    morphs = okt.pos(text, stem=True)  # stem=True -> 레마타이징 수행
    result = [word for word, tag in morphs if tag in ['Noun', 'Verb', 'Adjective']]
    return result


In [ ]:
df['cleaned'] = df['리뷰'].apply(clean_text)
df['tokens'] = df['cleaned'].apply(tokenize_and_lemmatize)
df

In [ ]:
from collections import Counter

def get_duplicates(token_list):
    counter = Counter(token_list)
    duplicates = {token: count for token, count in counter.items() if count > 1}
    return duplicates if duplicates else None

# 중복 단어 컬럼 추가
df['중복단어'] = df['tokens'].apply(get_duplicates)

In [ ]:
# ✅ 1. 중복 포함 빈도수
all_tokens = list(chain.from_iterable(df['tokens']))
counter_inclusive = Counter(all_tokens)
df_inclusive = pd.DataFrame(counter_inclusive.items(), columns=['토큰', '빈도수_중복포함'])

# ✅ 2. 중복 제거 후 빈도수
deduped_tokens = df['tokens'].apply(lambda x: list(set(x)))
all_deduped = list(chain.from_iterable(deduped_tokens))
counter_deduped = Counter(all_deduped)
df_deduped = pd.DataFrame(counter_deduped.items(), columns=['토큰', '빈도수_중복제거'])

# ✅ 3. 두 결과 병합
merged_df = pd.merge(df_inclusive, df_deduped, on='토큰', how='outer').fillna(0)
merged_df['빈도수_중복포함'] = merged_df['빈도수_중복포함'].astype(int)
merged_df['빈도수_중복제거'] = merged_df['빈도수_중복제거'].astype(int)
merged_df['차이'] = merged_df['빈도수_중복포함'] - merged_df['빈도수_중복제거']

# 결과 보기
merged_df.sort_values(by='빈도수_중복포함', ascending=False)

In [ ]:
df_inclusive.sort_values(by='빈도수_중복포함', ascending=False).head(10)

In [ ]:
df_inclusive.sort_values(by='빈도수_중복포함', ascending=False)['토큰'].head(30)

In [ ]:
df_deduped.sort_values(by='빈도수_중복제거', ascending=False).head(10)

In [ ]:
wordcloud_inclusive = WordCloud(
    background_color='white',
    width=800,
    height=400
).generate_from_frequencies(counter_inclusive)
wordcloud_inclusive

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# 워드클라우드 한글 폰트 지정 (예: 나눔고딕) — 시스템에 따라 경로 수정 필요
# Windows: 'malgun.ttf' / Mac: 'AppleGothic' / Linux: 'NanumGothic.ttf'
font_path = 'C:/Users/user/Projects/Busan_PublicData_Startup/assets/fonts/Pretendard-Medium.otf'
# plt 폰트 지정
plt.rcParams['font.family'] = 'Pretendard'

# 1. 중복 포함 워드클라우드
wordcloud_inclusive = WordCloud(
    font_path=font_path,
    background_color='white',
    width=800,
    height=400
).generate_from_frequencies(counter_inclusive)

# 2. 중복 제거 워드클라우드
wordcloud_deduped = WordCloud(
    font_path=font_path,
    background_color='white',
    width=800,
    height=400
).generate_from_frequencies(counter_deduped)

# 시각화
plt.figure(figsize=(8, 6))
plt.imshow(wordcloud_deduped, interpolation='bilinear')
plt.axis('off')
plt.title('워드클라우드 (중복 제거)')
# plt.subplot(1, 2, 1)
# plt.imshow(wordcloud_inclusive, interpolation='bilinear')
# plt.axis('off')
# plt.title('워드클라우드 (중복 포함)')

# plt.subplot(1, 2, 2)
# plt.imshow(wordcloud_deduped, interpolation='bilinear')
# plt.axis('off')
# plt.title('워드클라우드 (중복 제거)')

plt.tight_layout()
plt.show()
